# From Space to Action - Agricultural Drought Early Warning
## Notebook 01: Data Acquisition
**Goal:** Download satellite data from Google Earth Engine for Oromia pilot zone.

In [ ]:
!pip install -q earthengine-api geemap pyyaml tqdm

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/FromSpaceToAction'
DATA_DIR = f'{PROJECT_DIR}/data'
SRC_DIR = f'{PROJECT_DIR}/src'
MODELS_DIR = f'{PROJECT_DIR}/models'
OUTPUTS_DIR = f'{PROJECT_DIR}/outputs'
CONFIG_PATH = f'{PROJECT_DIR}/config/config.yaml'

for d in [DATA_DIR, f'{DATA_DIR}/raw', f'{DATA_DIR}/processed', f'{DATA_DIR}/features', f'{DATA_DIR}/targets', MODELS_DIR, OUTPUTS_DIR, f'{OUTPUTS_DIR}/maps', f'{OUTPUTS_DIR}/reports']:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, SRC_DIR)

In [ ]:
import ee
import geemap
import yaml
from tqdm.notebook import tqdm

try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

In [ ]:
# Define study area (Oromia pilot bbox from config)
bbox = [38.5, 7.5, 40.5, 9.5]
region = ee.Geometry.BBox(*bbox)
print(f"Study Area Defined: {bbox}")

### Section: Sentinel-2 - filter, preview collection size, export monthly composites

In [ ]:
s2 = ee.ImageCollection('COPERNICUS/S2_SR')\
    .filterBounds(region)\
    .filterDate('2018-01-01', '2018-02-01')\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

print("Sentinel-2 preview size:", s2.size().getInfo())

### Section: CHIRPS - daily rainfall, export monthly accumulations

In [ ]:
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')\
    .filterBounds(region)\
    .filterDate('2018-01-01', '2018-02-01')

print("CHIRPS preview size:", chirps.size().getInfo())

### Section: ERA5-Land - monthly temperature, humidity

In [ ]:
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')\
    .filterBounds(region)\
    .filterDate('2018-01-01', '2018-02-01')

print("ERA5-Land preview size:", era5.size().getInfo())

### Section: SMAP - soil moisture

In [ ]:
smap = ee.ImageCollection('NASA_USDA/HSL/SMAP10KM_soil_moisture')\
    .filterBounds(region)\
    .filterDate('2018-01-01', '2018-02-01')

print("SMAP preview size:", smap.size().getInfo())

### Section: DEM + Land Cover - static layers

In [ ]:
dem = ee.Image('USGS/SRTMGL1_003').clip(region)
landcover = ee.Image('ESA/WorldCover/v100/2020').clip(region)
print("Static layers loaded.")

### Section: Export tasks - batch export to Google Drive with progress tracking

In [ ]:
def export_image(image, description, folder, scale=1000):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        region=region.getInfo()['coordinates'],
        scale=scale,
        crs='EPSG:4326'
    )
    task.start()
    return task

# Example export: export_image(dem, 'Oromia_DEM', 'FromSpaceToAction_Exports')

### Section: FALLBACK - Generate sample data

In [ ]:
import pandas as pd
import numpy as np

def generate_sample_data(output_path):
    print("Generating sample data for demonstration...")
    dates = pd.date_range(start='2018-01-01', end='2025-12-31', freq='10D')
    lats = np.linspace(7.5, 9.5, 5)
    lons = np.linspace(38.5, 40.5, 5)
    
    data = []
    for d in dates:
        for lat in lats:
            for lon in lons:
                data.append({
                    'date': d,
                    'lat': lat,
                    'lon': lon,
                    'ndvi': np.random.uniform(0.1, 0.8),
                    'rainfall': np.random.uniform(0, 100),
                    'temperature': np.random.uniform(15, 35),
                    'soil_moisture': np.random.uniform(0, 40)
                })
    
    df = pd.DataFrame(data)
    df.to_parquet(output_path)
    print(f"Sample data saved to {output_path}")

sample_data_path = f'{DATA_DIR}/raw/sample_data.parquet'
generate_sample_data(sample_data_path)

### Summary

In [ ]:
print(f"Data Acquisition Complete.\nSample data location: {sample_data_path}")